In [12]:
import pandas as pd
import json
import numpy as np

In [6]:
import pandas as pd

# Lire le fichier CSV
df = pd.read_csv('/content/Amazon_review.csv', sep=',', encoding='utf-8')

# Afficher les 5 premières lignes
df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,"{'hi_res': array([None,\n 'https://m.med...","{'title': array([], dtype=object), 'url': arra...",Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",B01CUPMQZE,NaN,NaN,NaN
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,{'hi_res': array(['https://m.media-amazon.com/...,"{'title': array([], dtype=object), 'url': arra...",Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",B076WQZGPM,NaN,NaN,NaN
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,"{'hi_res': array([None, None], dtype=object), ...","{'title': array([], dtype=object), 'url': arra...",Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,NaN,NaN,NaN
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],NaN,{'hi_res': array(['https://m.media-amazon.com/...,"{'title': array([], dtype=object), 'url': arra...",Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",B088FKY3VD,NaN,NaN,NaN
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,['Material: 304 Stainless Steel; Brass tip'\n ...,['The Precision Plunger Bars are designed to w...,NaN,"{'hi_res': array([None], dtype=object), 'large...","{'title': array([], dtype=object), 'url': arra...",Precision,[],"{""UPC"": ""644287689178""}",B07NGFDN6G,NaN,NaN,NaN


Nettoyage & construction du jeu de données à traiter


- Exclure produits sans titre/store

- Parser details (JSON string)

- convertir price en float

- construire combined_features pour content-based

In [7]:
# nettoyage et colonnes dérivées
def safe_price(p):
    try:
        return float(p)
    except:
        return np.nan

def parse_details(d):
    if isinstance(d, str):
        try:
            j = json.loads(d)
            return ' '.join([f"{k} {v}" for k,v in j.items()])
        except:
            return ''
    return ''

df = df.dropna(subset=['title', 'store']).copy()
df['price'] = df['price'].apply(safe_price)
df['details_text'] = df['details'].apply(parse_details)
df['combined_features'] = (df['main_category'].fillna('') + ' ' +
                           df['title'].fillna('') + ' ' +
                           df['store'].fillna('') + ' ' +
                           df['details_text'].fillna(''))
# pour la suite, indexer clairement les items
df = df.reset_index(drop=True)
df['item_id'] = df.index
print("Après nettoyage :", len(df))


Après nettoyage : 101236


In [8]:
# échantillonner pour tests rapides
SAMPLE = 20000
if len(df) > SAMPLE:
    df = df.sample(SAMPLE, random_state=42).reset_index(drop=True)
    df['item_id'] = df.index
    print("Utilisation d'un échantillon de:", len(df))


Utilisation d'un échantillon de: 20000


In [9]:
# TF-IDF + similarité cosinus
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english', max_features=10000)  # max_features pour mémoire
tfidf_matrix = tfidf.fit_transform(df['combined_features'].values)
# calculer similarité (attention : full dense matrix peut être lourde)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)  # shape (N,N)


In [10]:
def recommend_content_by_title(title, top_k=10):
    mask = df['title'] == title
    if mask.sum() == 0:
        raise ValueError("Titre introuvable.")
    idx = df[mask].index[0]
    sims = list(enumerate(cosine_sim[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)
    top = [i for i,score in sims[1:top_k+1]]  # exclure lui-même
    return df.loc[top, ['item_id','title','store','price']].assign(score=[s for _,s in sims[1:top_k+1]])
